# AC-Zero — Annotation (Kaggle)

Annotates a grown group dataset with per-move-set distances
(`aczero dataset annotate`): each group's distance to the origin (the trivial
group) and its length-descent distance, per chosen move set. Writes only
`<name>.<moveset>.annotations.json` companion files to `/kaggle/working` — the
group dataset itself is downloaded read-only and never modified or re-uploaded.

This notebook is designed to run **in parallel** with `01_generate_dataset.ipynb`
in a separate Kaggle session: generation only ever appends groups, annotation
only ever reads the group graph and writes its own annotation files, so the two
never step on each other.

**Before you run**
- Settings -> **Internet: On** (needed to `pip install` from GitHub).
- Accelerator: **None / CPU** is enough — annotation is CPU graph search, no GPU.
- Kaggle sessions are capped at ~12 h. `TIME_BUDGET_HOURS` below stops each
  annotation pass with a safety margin so progress is always flushed to disk.
- Annotation is **resumable via the Hugging Face bucket**: any existing
  annotation file for a move set is pulled first as a warm start (`annotate()`
  only recomputes groups whose shorter-distance is still unresolved), then the
  updated file is pushed back at the end.


## 1. Configuration

In [ ]:
import os

# --- Repository -------------------------------------------------------------
REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"  # branch / tag / commit to install

# --- Dataset to annotate -----------------------------------------------------
RANK = 2  # group rank; must match the grown dataset
DATASET_NAME = f"train_rank{RANK}.groups.json"
WORKERS = 0  # 0 = all CPU cores; 1 = single in-process worker

# --- Annotation passes --------------------------------------------------------
# Move sets to annotate, in order. Each writes its own companion
# <name>.<moveset>.annotations.json and never touches the group dataset. Default
# is "universal" (the full invertible catalog) plus "strict-ac" (the classic
# primitive AC moves: multiply, invert, conjugate) -- strict-ac is what
# training's `descent` reward (02_train, soon 03_train, REWARD_MODE = "descent")
# reads its descent distance from.
ANNOTATE_MOVESETS = ["universal", "strict-ac"]
MAX_DEPTH = 32  # max moves per per-group shorter-distance search
CHECKPOINT_EVERY = 5000  # rewrite the annotation file every N freshly settled groups

# --- Time budget (Kaggle hard-kills at ~12 h) -------------------------------
TIME_BUDGET_HOURS = 11.5  # stop the current pass after this much wall-clock
SAFETY_MARGIN_MIN = 10  # head-room reserved for the final flush

# --- Output -------------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

# --- Hugging Face bucket (dataset storage; too big for GitHub) ---------------
# Pulls the group dataset read-only (this notebook never uploads it back -- only
# 01_generate_dataset does) and any existing annotation file per move set for a
# warm start. Add an `HF_TOKEN` Kaggle secret (Add-ons -> Secrets) with write
# access to the bucket namespace.
HF_BUCKET = "HkHk2Prod/alphaac-data"
HF_DOWNLOAD_ON_START = True
HF_UPLOAD_ON_FINISH = True

## 2. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys


# --no-deps keeps Kaggle's pre-built numpy; we add only the light deps the
# annotation path actually needs (it is pure-Python graph search, no torch).
def pip_install(*args):
    # Quiet install: capture pip's output and surface it only on real failure.
    # Kaggle's base image ships a broken google-adk dependency, so pip prints a
    # "dependency resolver" ERROR on every run even when our install succeeds;
    # capturing keeps that unrelated noise (and download progress bars) out of
    # the log while still raising if the install itself fails.
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True,
        text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n")
        proc.check_returncode()


spec = f"git+{REPO_URL}@{REPO_BRANCH}"
pip_install("--no-deps", spec)
pip_install(
    "numpy>=1.26",
    "pydantic>=2",
    "pyyaml>=6",
    "typer>=0.12",
    "rich>=13",
    "gymnasium>=0.29",
    "huggingface_hub>=1.21",
)

print("Installed ac_zero from", spec)

## 3. Time budget

In [ ]:
import time

START_TIME = time.monotonic()
DEADLINE = START_TIME + TIME_BUDGET_HOURS * 3600
MARGIN_S = SAFETY_MARGIN_MIN * 60


def elapsed_s():
    return time.monotonic() - START_TIME


def time_left_s():
    return DEADLINE - time.monotonic()


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"


class DeadlineReached(Exception):
    """Raised from a progress callback to stop cleanly before the Kaggle kill."""


print(
    f"Budget {TIME_BUDGET_HOURS} h with a {SAFETY_MARGIN_MIN} min safety margin "
    f"(stop annotating when {hms(MARGIN_S)} remain)."
)

## 4. Download the dataset (read-only) and warm-start existing annotations

Pulls the group dataset grown by `01_generate_dataset.ipynb` from the bucket --
this notebook only ever reads it. For each move set in `ANNOTATE_MOVESETS`, also
pulls its existing annotation file if one is already in the bucket, so the pass
below resumes instead of recomputing settled groups from scratch.

In [ ]:
import os
from pathlib import Path

from ac_zero.datasets.annotate import annotation_path
from ac_zero.datasets.hub import download_dataset

# Pick up the HF token from a Kaggle secret if it is not already in the env.
if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass  # no HF_TOKEN secret attached to this kernel

if HF_UPLOAD_ON_FINISH and not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_UPLOAD_ON_FINISH is True but no HF_TOKEN is available (env var or "
        "Kaggle secret) -- annotating would run for the whole time budget and "
        "then fail to publish the result. Add an HF_TOKEN Kaggle secret "
        "(Add-ons -> Secrets) with write access, or set HF_UPLOAD_ON_FINISH = False."
    )

dataset_path = Path(OUTPUT_DIR) / DATASET_NAME
dataset_path.parent.mkdir(parents=True, exist_ok=True)

if HF_DOWNLOAD_ON_START:
    pulled = download_dataset(
        dataset_path, remote_name=DATASET_NAME, bucket=HF_BUCKET, missing_ok=True
    )
    if pulled is None:
        raise FileNotFoundError(
            f"No {DATASET_NAME!r} in hf://buckets/{HF_BUCKET} yet -- "
            "run 01_generate_dataset.ipynb first."
        )
    print(f"Annotating {pulled} ({pulled.stat().st_size / 1e6:.1f} MB), read-only.")
elif not dataset_path.exists():
    raise FileNotFoundError(f"{dataset_path} not found and HF_DOWNLOAD_ON_START is False.")

for moveset in ANNOTATE_MOVESETS:
    apath = annotation_path(dataset_path, moveset)
    if not HF_DOWNLOAD_ON_START:
        continue
    existing = download_dataset(apath, remote_name=apath.name, bucket=HF_BUCKET, missing_ok=True)
    if existing is None:
        print(f"No existing {moveset} annotations in the bucket -- starting fresh.")
    else:
        print(f"Warm start: resumed {existing.name} from the bucket.")

## 5. Annotate

Runs `aczero dataset annotate` for each move set in `ANNOTATE_MOVESETS`, in
order. Each pass is time-boxed like the generation notebook's grow loop: it
checkpoints periodically, so an interrupted pass leaves a resumable, partially
settled annotation file rather than losing progress.

In [ ]:
from ac_zero.datasets.annotate import AnnotateConfig, annotate

annotation_paths = []
if not ANNOTATE_MOVESETS:
    print("Nothing to do: ANNOTATE_MOVESETS is empty.")
for moveset in ANNOTATE_MOVESETS:
    acfg = AnnotateConfig(
        moveset=moveset, max_depth=MAX_DEPTH, workers=WORKERS, checkpoint_every=CHECKPOINT_EVERY
    )
    apath = annotation_path(dataset_path, moveset)

    def aprogress(message, metrics, moveset=moveset):
        if time_left_s() <= MARGIN_S:
            raise DeadlineReached(message)
        print(f"[{hms(elapsed_s())}] {moveset} {message}: {metrics}")

    try:
        arep = annotate(dataset_path, acfg, progress=aprogress)
        annotation_paths.append(apath)
        print(f"annotate[{moveset}]:", arep)
    except DeadlineReached:
        annotation_paths.append(apath)
        print(
            f"[{hms(elapsed_s())}] {moveset} annotation stopped at the time budget "
            "(partial, checkpointed)."
        )
        break

print("\nAnnotation files:", [str(p) for p in annotation_paths])

## 6. Publish annotations to the Hugging Face bucket

In [ ]:
if HF_UPLOAD_ON_FINISH:
    from ac_zero.datasets.hub import upload_dataset

    for path in annotation_paths:
        if not path.exists():
            continue
        try:
            uri = upload_dataset(path, remote_name=path.name, bucket=HF_BUCKET)
            print("Uploaded", path.name, "to", uri)
        except Exception as exc:
            print("Bucket upload skipped for", path.name, ":", exc)
else:
    print("HF_UPLOAD_ON_FINISH is False - annotations left in", OUTPUT_DIR)

print("\nAnnotation artifacts in", OUTPUT_DIR, ":")
for p in sorted(Path(OUTPUT_DIR).glob("*.annotations.json")):
    print(f"  {p.name:40s} {p.stat().st_size / 1e3:9.1f} KB")